### Enabling GPU Acceleration (Optional)

In [1]:
import tensorflow as tf

print("GPU Available:", tf.config.list_physical_devices('GPU'))

# Use the first GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("GPU is enabled!")
    except RuntimeError as e:
        print(e)

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU is enabled!


### Loading and Splitting the Dataset

In [2]:
!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2008/VOCtrainval_14-Jul-2008.tar

--2025-03-15 21:51:39--  http://host.robots.ox.ac.uk/pascal/VOC/voc2008/VOCtrainval_14-Jul-2008.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.152
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.152|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 577034240 (550M) [application/x-tar]
Saving to: ‘VOCtrainval_14-Jul-2008.tar’

VOCtrainval_14-Jul- 100%[===================>] 550.30M  13.6MB/s    in 43s     

2025-03-15 21:52:23 (12.7 MB/s) - ‘VOCtrainval_14-Jul-2008.tar’ saved [577034240/577034240]



In [3]:
!tar -xvf VOCtrainval_14-Jul-2008.tar

VOCdevkit/
VOCdevkit/VOC2008/
VOCdevkit/VOC2008/Annotations/
VOCdevkit/VOC2008/Annotations/2007_000027.xml
VOCdevkit/VOC2008/Annotations/2007_000032.xml
VOCdevkit/VOC2008/Annotations/2007_000033.xml
VOCdevkit/VOC2008/Annotations/2007_000039.xml
VOCdevkit/VOC2008/Annotations/2007_000042.xml
VOCdevkit/VOC2008/Annotations/2007_000061.xml
VOCdevkit/VOC2008/Annotations/2007_000063.xml
VOCdevkit/VOC2008/Annotations/2007_000068.xml
VOCdevkit/VOC2008/Annotations/2007_000121.xml
VOCdevkit/VOC2008/Annotations/2007_000123.xml
VOCdevkit/VOC2008/Annotations/2007_000129.xml
VOCdevkit/VOC2008/Annotations/2007_000170.xml
VOCdevkit/VOC2008/Annotations/2007_000175.xml
VOCdevkit/VOC2008/Annotations/2007_000187.xml
VOCdevkit/VOC2008/Annotations/2007_000241.xml
VOCdevkit/VOC2008/Annotations/2007_000243.xml
VOCdevkit/VOC2008/Annotations/2007_000250.xml
VOCdevkit/VOC2008/Annotations/2007_000256.xml
VOCdevkit/VOC2008/Annotations/2007_000272.xml
VOCdevkit/VOC2008/Annotations/2007_000323.xml
VOCdevkit/VOC2008/A

The code processes XML files containing object annotations for images, extracts the bounding box co-ordinates for each object, and crops the images accordingly. I then assigned the cropped images to  their respective class directories, based on the object class name (e.g., "dog," "cat"). The cropped images are saved with unique filenames in the appropriate class subdirectories, creating an organized dataset.

In [4]:
import os
import xml.etree.ElementTree as et
from PIL import Image

count = 0
output_path = "dataset/"

def process(xml):
    global count
    global output_path
    tree = et.parse(xml)
    root = tree.getroot()

    filename = root.find("filename").text

    for obj in root.findall("object"):
        img_path = "/kaggle/working/VOCdevkit/VOC2008/JPEGImages"

        name = obj.find("name").text
        bndbox = obj.find('bndbox')
        xmin = int(round(float(bndbox.find('xmin').text)))
        ymin = int(round(float(bndbox.find('ymin').text)))
        xmax = int(round(float(bndbox.find('xmax').text)))
        ymax = int(round(float(bndbox.find('ymax').text)))

        image_path = os.path.join(img_path, filename)
        image = Image.open(image_path)
        cropped_image = image.crop((xmin, ymin, xmax, ymax))

        class_path = os.path.join(output_path, name)
        os.makedirs(class_path, exist_ok=True)

        cropped_image.save(os.path.join(class_path, f"{count}.jpg"))

        count += 1


def process_all(xml_path):
    for xml in os.listdir(xml_path):
        process(os.path.join(xml_path, xml))

In [5]:
process_all("/kaggle/working/VOCdevkit/VOC2008/Annotations")

I then split the classes into Train, Val and Test sets, and organized them in a way so that they can be extracted using TensorFlow later. The dataset is split in a 70/15/15 ratio.

In [6]:
import random
import shutil

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

for class_name in os.listdir(output_path):
    class_path = os.path.join(output_path, class_name)

    if os.path.isdir(class_path):
        image_files = [f for f in os.listdir(class_path)]

        random.shuffle(image_files)

        num_images = len(image_files)
        train_end = int(train_ratio * num_images)
        val_end = train_end + int(val_ratio * num_images)

        train_folder = os.path.join(output_path, 'Train', class_name)
        val_folder = os.path.join(output_path, 'Val', class_name)
        test_folder = os.path.join(output_path, 'Test', class_name)

        os.makedirs(train_folder, exist_ok=True)
        os.makedirs(val_folder, exist_ok=True)
        os.makedirs(test_folder, exist_ok=True)

        for i, img_file in enumerate(image_files):
            src_path = os.path.join(class_path, img_file)
            if i < train_end:
                dst_path = os.path.join(train_folder, img_file)
            elif i < val_end:
                dst_path = os.path.join(val_folder, img_file)
            else:
                dst_path = os.path.join(test_folder, img_file)

            shutil.move(src_path, dst_path)

### Pre-Processing the Dataset

The only form of data preprocessing I applied was rescaling, to improve performance. Then, I extracted the data into their appropriate splits using TensorFlow

In [7]:
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [8]:
train_path = "dataset/Train"
val_path = "dataset/Val"
test_path = "dataset/Test"

image_size = (180, 180)
batch_size = 64

train_datagen = ImageDataGenerator(
    rescale=(1./255)
)
val_datagen = ImageDataGenerator(
    rescale=(1./255)
)
test_datagen = ImageDataGenerator(
    rescale=(1./255)
)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)

test_data = test_datagen.flow_from_directory(
    test_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)

Found 10273 images belonging to 20 classes.
Found 2192 images belonging to 20 classes.
Found 2224 images belonging to 20 classes.


### Training the Models

I will be using VGG16, InceptionV3, and MobileNetV2 models. I will be adding the same 3 layers to all the model, 2 Dense layers with 256 and 128 neurons respectively, and then an output layer with 20 neurons (for 20 classes). Since, we are performing transfer learning and not fine-tuning, I froze all the layers in the base model, so that only the parameters of the added layers are trained.

In [9]:
from tensorflow.keras.applications import VGG16, MobileNetV2, InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, GlobalAveragePooling2D
import tensorflow.keras as tf

#### VGG16

In [10]:
vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(180, 180, 3))
vgg_base.trainable = False # freezing the weights

model = Sequential([
        vgg_base,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(20, activation='softmax')  # 20 classes
    ])

model.compile(
    optimizer=tf.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 5, 5, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 20)                  │           2,580 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,881,492 (56.77 MB)

 Trainable params: 166,804 (651.58 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [12]:
history_vgg = model.fit(train_data, validation_data=val_data, epochs=15)

Epoch 1/15


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


161/161 ━━━━━━━━━━━━━━━━━━━━ 40s 164ms/step - accuracy: 0.3660 - loss: 2.4839 - val_accuracy: 0.4033 - val_loss: 2.1040
Epoch 2/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 107ms/step - accuracy: 0.4198 - loss: 2.0075 - val_accuracy: 0.4658 - val_loss: 1.8282
Epoch 3/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 106ms/step - accuracy: 0.4679 - loss: 1.7880 - val_accuracy: 0.5160 - val_loss: 1.6674
Epoch 4/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 106ms/step - accuracy: 0.5232 - loss: 1.6269 - val_accuracy: 0.5447 - val_loss: 1.5540
Epoch 5/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 108ms/step - accuracy: 0.5556 - loss: 1.5032 - val_accuracy: 0.5716 - val_loss: 1.4754
Epoch 6/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 107ms/step - accuracy: 0.5753 - loss: 1.4400 - val_accuracy: 0.5880 - val_loss: 1.4230
Epoch 7/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 107ms/step - accuracy: 0.5957 - loss: 1.3676 - val_accuracy: 0.5963 - val_loss: 1.3783
Epoch 8/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 18s 107ms/step - accuracy: 0.6043 - loss: 1.3235 - val

In [13]:
vgg_test_acc = model.evaluate(test_data)

35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 304ms/step - accuracy: 0.6505 - loss: 1.1593
